In [0]:
%run ./common/_utils_config

In [0]:
%run ./common/_utils_reference_data

In [0]:
%run ./common/_utils_bronze_framework

In [0]:
inventory = [describe_source_footprint(name) for name in list_source_specs()]

display(spark.createDataFrame(
    [(i["source_name"], len(i["endpoints"]), i["raw_files"], len(i["bronze_tables"]), i["log_rows"]) for i in inventory],
    "source_name STRING, endpoints INT, raw_files INT, bronze_tables INT, log_rows BIGINT",
).orderBy("source_name"))

In [0]:
for source_name in ["open_meteo", "open_meteo_forecast_malaga", "open_meteo_forecast_seville"]:
    decommission_source(source_name)

In [0]:
print("Remaining sources:", list_source_specs())

display(
    spark.table(ingestion_log_table)
    .filter(F.col("status") == "decommissioned")
    .select("source_name", "endpoint_name", "status", "file_path", "error_message", "ingested_at")
    .orderBy(F.desc("ingested_at"))
)

In [0]:
source_name = "open_meteo_archive"
spec = load_source_spec_file(source_name)

errors = validate_source_spec(spec)
print("✅ Spec valid" if not errors else f"❌ {errors}")

summary = run_source_spec(spec, days_back_override=6, dry_run=True)
for r in summary["results"]:
    print(f"{r['chunk_start']}->{r['chunk_end']}: {r['status']} · {r['top_level_items']} location blocks")

In [0]:
summary = run_source_spec(spec)
print(f"\n{summary['chunks_ok']} ok / {summary['chunks_error']} errors · tables: {list(summary['bronze_tables'])}")

In [0]:
table = bronze_table_name(source_name, "hourly_weather")

display(
    spark.table(table)
    .select("chunk_start", "chunk_end",
            F.size("location_codes").alias("locations"),
            F.expr("array_size(cast(payload as array<variant>))").alias("payload_blocks"),
            F.element_at("location_codes", 1).alias("first_code"),
            F.expr("payload:[0].latitude::double").alias("first_lat"),
            F.expr("payload:[0].hourly.time[0]::string").alias("first_hour_utc"),
            F.expr("payload:[0].hourly.wind_speed_100m[0]::double").alias("first_wind_100m"))
    .orderBy("chunk_start")
)